In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
# Write your code here


import torch
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import os
from torch.utils.data import Dataset
from PIL import Image
import glob


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



# Define transformations
# secind we do Transforms
train_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomRotation(15),   #required augmentation
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])


In [ ]:


# ImageFolder if applicable here so im gonna use it since it is easier
data_dir = path
print("Dataset folders:", os.listdir(data_dir))

full_dataset = datasets.ImageFolder(root=data_dir, transform=train_transform)

class_names = full_dataset.classes
print("Classes:", class_names)
print("Total images:", len(full_dataset))

In [ ]:
# training and testing sizes are not defind yet so
# i did them hereas 80% for training and the rest ofr testing
from torch.utils.data import  random_split

train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])


#i did this part so test dataset doesnt get augmented and
# not just memorise random noise which cause it to overfit
test_dataset.dataset.transform = test_transform

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))



"""
important note:

I couldv'e done it like Mohammed ALtaib way he said in his video u can do this:

train_end = int(0.8 * N)
valid_end = int(0.9 * N)

train_image_paths = image_paths[:train_end]
valid_image_paths = image_paths[train_end:valid_end]
test_image_paths  = image_paths[valid_end:]


but i find mine to be more useful
"""

In [ ]:
# my data Loaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:

# displaying images
import matplotlib.pyplot as plt
import numpy as np
# Get a batch of training data
data_iter = iter(train_loader)
images, labels = next(data_iter)

# Potato disease class names
classes = class_names

# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))  # (C, H, W) -> (H, W, C)

    ax.imshow(img)
    ax.set_title(classes[labels[i].item()])
    ax.axis("off")

plt.show()



In [ ]:
# Write your code here
import torch.nn as nn
import torch

# Define the CNN Model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()


        # Convolutional Layers
        # ill explain one line then just keep going 5 lines with no explanation
        # here a conv2d layers that is fully colored becaue of the 3 channels
      # dwill get convolved normally then batch normalized after each layer(the batch normlization is the output of previous layer)
      # all of this is before
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(16)


        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(32)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3   = nn.BatchNorm2d(64)

        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn4   = nn.BatchNorm2d(128)

        self.conv5 = nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, padding=1)
        self.bn5   = nn.BatchNorm2d(256)


        # here im using the relu activation
        # Activation
        self.relu = nn.ReLU()

        # Pooling Layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)


        # Fully Connected Layers
        self.fc1 = nn.Linear(256 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 3)  # 3 classes: Early_blight, Late_blight, healthy

    def forward(self, x):
        # Conv Block 1
        x = self.pool(self.relu(self.bn1(self.conv1(x))))
        x = self.pool(self.relu(self.bn2(self.conv2(x))))
        x = self.pool(self.relu(self.bn3(self.conv3(x))))

        x = self.relu(self.bn4(self.conv4(x)))
        x = self.relu(self.bn5(self.conv5(x)))


        # Flatten
        x = x.view(x.size(0), -1)

        # Fully Connected
        x = self.relu(self.fc1(x))
        x = self.fc2(x)   # Logits (no softmax)

        return x

In [ ]:
# Write your code here



# Write your code here
from tqdm import tqdm
import torch

# Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Set model to training mode

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Accuracy calculation
        predictions = outputs.argmax(dim=1)  # No Softmax needed
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Accuracy in percentage

    return avg_loss, accuracy

# Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()  # Set model to evaluation mode

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient computation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()

            # Accuracy calculation
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total  # Compute accuracy in percentage

    return avg_loss, accuracy


In [ ]:
# Write your code here
import torch
import torch.optim as optim

# Initialize the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CNNModel().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()  # Multi-class Classification loss (Input: Logits, not probabilities)
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Adam optimizer
num_epochs = 10  # Number of epochs

# Lists to store metrics
train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

# Training process
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_accuracy = validate(
        model, test_loader, criterion, device
    )

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Accuracy={train_accuracy:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Accuracy={val_accuracy:.2f}%")

In [ ]:
import matplotlib.pyplot as plt


# Plot Training & Validation Loss
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()



# Plot Training & Validation Accuracy
plt.figure(figsize=(8, 4))
plt.plot(train_accuracies, label="Training Accuracy")
plt.plot(val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()
plt.show()



#wow
#there is something so wrong in this model
#if time allows ill try to comeback and debug what happenes i only got 58 mins
#    ):

In [ ]:
# Write your code here


In [ ]:
### NO TIIIIIIIIIIME  ):